In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import lightgbm as lgb

SEED = 42

In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

SEED = 42
np.random.seed(SEED)

test_ids = test['id']

train = train.drop(columns=['id'])
test = test.drop(columns=['id'])

In [3]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()
    
    # فیچرهای اصلی و مفید (بر اساس کد اولیه‌ات + کمی بهبود بدون overfit)
    data['total_bees'] = data['honeybee'] + data['bumbles'] + data['andrena'] + data['osmia']
    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['osmia_honeybee_inter'] = data['osmia'] * data['honeybee']
    data['wild_bees'] = data['bumbles'] + data['andrena'] + data['osmia']
    
    data['temp_range'] = data['MaxOfUpperTRange'] - data['MinOfLowerTRange']
    data['avg_temp'] = (data['AverageOfUpperTRange'] + data['AverageOfLowerTRange']) / 2
    data['temp_x_rain'] = data['avg_temp'] * data['RainingDays']
    data['temp_range_x_rain'] = data['temp_range'] * data['RainingDays']
    
    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)
    data['fruitmass_x_seeds'] = data['fruitmass'] * data['seeds']
    
    # Non-linear ساده (log و sqrt کافی است، sq باعث overfit شد)
    for col in ['clonesize', 'total_bees', 'fruitmass', 'seeds', 'fruit_seed_ratio']:
        data[f'log_{col}'] = np.log1p(data[col])
        data[f'sqrt_{col}'] = np.sqrt(data[col])
    
    # Clustering متوسط (8 cluster + بدون one-hot برای جلوگیری از overfit)
    cluster_cols = ['clonesize', 'total_bees', 'avg_temp', 'RainingDays', 'fruitmass', 'seeds']
    if fit:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(data[cluster_cols])
        kmeans = KMeans(n_clusters=8, random_state=SEED, n_init=30)
        data['cluster'] = kmeans.fit_predict(scaled)
    else:
        scaled = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(scaled)
    
    return data, kmeans, scaler

train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)

X = train_fe.drop(columns=['yield'])
y = train_fe['yield']

In [4]:
params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'learning_rate': 0.02,              # تعادل بین دقت و سرعت
    'num_leaves': 100,                  # بیشتر از اولیه، کمتر از overfit
    'min_data_in_leaf': 30,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'lambda_l1': 0.3,
    'lambda_l2': 0.7,
    'min_gain_to_split': 0.005,
    'boosting': 'gbdt',
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'max_depth': -1,
    'verbosity': -1,
    'seed': SEED
}

kf = KFold(n_splits=10, shuffle=True, random_state=SEED)
oof = np.zeros(len(X))
test_preds = np.zeros(len(test_fe))
maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f'Fold {fold}/10')
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    train_set = lgb.Dataset(X_tr, y_tr)
    val_set = lgb.Dataset(X_val, y_val, reference=train_set)
    
    model = lgb.train(
        params,
        train_set,
        num_boost_round=8000,                   # بالا اما با early stopping قوی
        valid_sets=[train_set, val_set],
        callbacks=[
            lgb.early_stopping(stopping_rounds=400, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )
    
    val_pred = model.predict(X_val)
    oof[val_idx] = val_pred
    fold_mae = mean_absolute_error(y_val, val_pred)
    maes.append(fold_mae)
    print(f'   Fold {fold} MAE: {fold_mae:.5f}')
    
    test_preds += model.predict(test_fe) / kf.n_splits

print('\nFinal CV OOF MAE:', np.mean(maes), '±', np.std(maes))

Fold 1/10


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


   Fold 1 MAE: 238.89059
Fold 2/10
   Fold 2 MAE: 259.49872
Fold 3/10
   Fold 3 MAE: 248.65429
Fold 4/10
   Fold 4 MAE: 243.00583
Fold 5/10
   Fold 5 MAE: 244.46371
Fold 6/10
   Fold 6 MAE: 239.14323
Fold 7/10
   Fold 7 MAE: 250.99691
Fold 8/10
   Fold 8 MAE: 243.26996
Fold 9/10
   Fold 9 MAE: 242.55010
Fold 10/10
   Fold 10 MAE: 245.34586

Final CV OOF MAE: 245.58191939446232 ± 5.840794367380928


In [5]:
final_test_preds = np.clip(test_preds, y.min(), y.max())

submission = pd.DataFrame({
    'id': test_ids,
    'yield': final_test_preds
})
submission.to_csv('submission.csv', index=False)
print("\nSubmission head:")
print(submission.head())


Submission head:
      id        yield
0  15000  7509.770703
1  15001  5887.964590
2  15002  6471.911430
3  15003  4656.618245
4  15004  5895.838907
